# Skan Tests

Developing tests for `skan.csr.iteratively_prune_paths` by generating some [random shapes]() and then skeletonising them

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import napari
import numpy as np
from numpy.testing import assert_equal, assert_almost_equal
from skimage.draw import random_shapes
from skimage.morphology import skeletonize, label
from skan import csr, iteratively_prune_paths, summarize, Skeleton
from skan._testdata import tinycycle, tinyline, skeleton0, skeleton1, skeleton2, skeleton3d, topograph1d, skeleton4

## Functions

In [ ]:
def random_skeleton(**kwargs) -> np.ndarray:
    """Generate a random skeleton using skimage.draw.random_shapes()

    Parameters
    ----------
    **kwargs : dict
         Dictionary of arguments used by skimage.draw.random_shapes()

    Returns
    -------
    np.ndarray
        2D array of a skeleton.
    """
    random_images, _ = random_shapes(**kwargs)
    mask = np.where(random_images != 255, 1, 0)
    skeleton = skeletonize(mask)
    return skeleton

In [ ]:
def napari_prep(skeleton: Skeleton):
    """Function for preparing to plot a skan skeleton with napari.

    Parameters
    ----------

    skeleton: Skeleton
        skan.Skeleton which has edges and paths defined.

    Returns
    -------
    tuple: all_paths, paths_table
        all_paths is the co-ordinates for all paths/edges in the plot, paths_table is a summary of the skeleton,
        augmented with random colours for each edge.
    """
    paths_table = summarize(skeleton)
    all_paths = [skeleton.path_coordinates(i) for i in range(skeleton.n_paths)]
    paths_table["random-path-id"] = np.random.default_rng().permutation(skeleton.n_paths)
    return all_paths, paths_table


def napari_plot(skeleton: Skeleton, ndisplay: int = 2) -> None:
    """Plot a skeleton using Napari

    Creates and launches a new napari.Viewer() and adds the skeleton to the plot.

    Parameters
    ----------

    skeleton: Skeleton
        skan.Skeleton which has edges and paths defined.
    ndisplay: int
        Number of dimensions to display.
    """
    all_paths, paths_table = napari_prep(skeleton)
    viewer = napari.Viewer(ndisplay=ndisplay)
    skeleton_layer = viewer.add_shapes(
        all_paths,
        shape_type="path",
        properties=paths_table,
        edge_width=0.5,
        edge_color="random_path_id",
        edge_colormap="tab10",
    )

## Artificial Skeletons

We make some artificial skeletons using `random_shapes()` which often overlap and are then skeletonised

| seed        | min_size |Description                         |
|-------------|----------|------------------------------------|
| 1           |    20    | Loop with multiple sidebranches    |
| 21          |    20    | Small fork at one end of linear    |
| 165103      |    60    | Loop with multiple sidebranches    |
| 894632511   |    20    | Multiple skeletons in single image |
| 13588686514 |    20    | linear with lots of side-branches  |


In [ ]:
kwargs = {
    "image_shape": (128, 128),
    "max_shapes": 20,
    "channel_axis": None,
    "shape": None,
    "rng": 1,  # 351381894132554,
    "allow_overlap": True,
    "min_size": 20,
}
skeleton_loop = random_skeleton(**kwargs)
plt.imshow(skeleton_loop)

In [ ]:
napari_plot(Skeleton(skeleton_loop))

In [ ]:
pruned_skeleton = iteratively_prune_paths(skeleton_loop, imgname="loopy")
plt.imshow(pruned_skeleton.skeleton_image)

In [ ]:
kwargs = {
    "image_shape": (128, 128),
    "max_shapes": 20,
    "channel_axis": None,
    "shape": None,
    "rng": 165103,
    "allow_overlap": True,
    "min_size": 60,
}
skeleton_loop2 = random_skeleton(**kwargs)
plt.imshow(skeleton_loop2)

In [ ]:
pruned_skeleton_loop2 = iteratively_prune_paths(skeleton_loop2, imgname="loopy")
plt.imshow(pruned_skeleton_loop2.skeleton_image)

In [ ]:
kwargs = {
    "image_shape": (128, 128),
    "max_shapes": 20,
    "channel_axis": None,
    "shape": None,
    "rng": 13588686514,
    "allow_overlap": True,
    "min_size": 20,
}
skeleton_linear1 = random_skeleton(**kwargs)
plt.imshow(skeleton_linear1)

In [ ]:
pruned_skeleton_linear1 = iteratively_prune_paths(skeleton_linear1)
plt.imshow(pruned_skeleton_linear1.skeleton_image)

In [ ]:
kwargs = {
    "image_shape": (128, 128),
    "max_shapes": 20,
    "channel_axis": None,
    "shape": None,
    "rng": 21,
    "allow_overlap": True,
    "min_size": 20,
}
skeleton_linear2 = random_skeleton(**kwargs)
plt.imshow(skeleton_linear2)

In [ ]:
pruned_skeleton_linear2 = iteratively_prune_paths(skeleton_linear2)
plt.imshow(pruned_skeleton_linear2.skeleton_image)

In [ ]:
kwargs = {
    "image_shape": (128, 128),
    "max_shapes": 20,
    "channel_axis": None,
    "shape": None,
    "rng": 894632511,
    "allow_overlap": True,
    "min_size": 20,
}
skeleton_linear3 = random_skeleton(**kwargs)
plt.imshow(skeleton_linear3)

In [ ]:
pruned_skeleton_linear3 = iteratively_prune_paths(skeleton_linear3)
plt.imshow(pruned_skeleton_linear3.skeleton_image)